In [1]:
import pyspark
from pyspark.sql import types,SparkSession
import pandas as pd
import re,os

In [2]:
jardrv = "/Users/snehil/Downloads/postgresql-42.7.5.jar"

spark = SparkSession.builder.config('spark.driver.extraClassPath', jardrv).getOrCreate()
url = 'jdbc:postgresql://127.0.0.1/coffee'
properties = {'user': os.getenv('POSTGRES_LOCAL_USER'), 'password': os.getenv('POSTGRES_LOCAL_PASSWORD')}
df = spark.read.jdbc(url=url, table='public.transformed_stg', properties=properties)

25/03/24 01:22:16 WARN Utils: Your hostname, Snehils-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.29.183 instead (on interface en0)
25/03/24 01:22:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/24 01:22:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# print dataframe
df.show()

+---------------+--------------------+--------------------+------+--------+---------------+--------------------+-------------------+------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+-----+-------+-----------------+--------------------+
|        roaster|                name|                link| price|altitude|       varietal|          processing|             estate| roast_level|       tasting_notes|         description|country|          scraped_at|      transformed_at|            location|           producers|aroma|acidity|             body|    other_properties|
+---------------+--------------------+--------------------+------+--------+---------------+--------------------+-------------------+------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+-----+-------+-----------------+--------------------+
|

In [4]:
# print schema
df.printSchema()

root
 |-- roaster: string (nullable = true)
 |-- name: string (nullable = true)
 |-- link: string (nullable = true)
 |-- price: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- varietal: string (nullable = true)
 |-- processing: string (nullable = true)
 |-- estate: string (nullable = true)
 |-- roast_level: string (nullable = true)
 |-- tasting_notes: string (nullable = true)
 |-- description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- scraped_at: timestamp (nullable = true)
 |-- transformed_at: timestamp (nullable = true)
 |-- location: string (nullable = true)
 |-- producers: string (nullable = true)
 |-- aroma: string (nullable = true)
 |-- acidity: string (nullable = true)
 |-- body: string (nullable = true)
 |-- other_properties: string (nullable = true)



In [5]:
# create a temporary view
df.createOrReplaceTempView('coffee')

In [6]:
df_sql = spark.sql("""

    select roaster, max(price) as expensive_coffee, min(price) as cheapest_coffee, count(*) as coffee_count from coffee group by roaster
    
""")

In [7]:
df_sql.show()

+--------------------+----------------+---------------+------------+
|             roaster|expensive_coffee|cheapest_coffee|coffee_count|
+--------------------+----------------+---------------+------------+
|           fraction9|          1149.0|          349.0|          26|
|          savorworks|           825.0|          275.0|          13|
|        curious_life|          3900.0|          550.0|          10|
|          blue_tokai|          2000.0|          250.0|          46|
|          half_light|           600.0|          475.0|           5|
|      corridor_seven|           910.0|          440.0|          20|
|         kc_roasters|           750.0|          750.0|          12|
|            greysoul|           749.0|          399.0|          22|
|bloom_coffee_roas...|           775.0|          250.0|          19|
|     koffie_genetics|           725.0|          300.0|          16|
|         kapi_kottai|          8670.0|          420.0|          21|
|               naivo|          13